# 🎙️ ProXMax — English → Bangla ভিডিও ডাবিং

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/redwan-primeAmax/ProXMax/blob/main/notebook/ProXMax_Colab.ipynb)

> 📌 **`Runtime → Run all` চাপলেই কাজ শেষ।** আগে `Runtime → Change runtime type → T4 GPU`
> সেট করে নিন। **টার্মিনালে কিছু টাইপ করতে হবে না** — শুধু **ধাপ ৬**-এর সেলে
> আপনার Gemini API key পেস্ট করবেন। কণ্ঠের জন্য কিছুই লাগে না (বিল্ট-ইন বাংলা ভয়েস)।

এই নোটবুকটি Google Drive-এর `Video/input` ফোল্ডারের **সব English ভিডিও** একে একে বাংলায় ডাবিং করে `Video/output`-এ রাখে।

**এই সংস্করণে নতুন কী:**

- 🎤 **কিছুই দিতে হয় না** — বিল্ট-ইন `bn-BD` neural ভয়েস; নিজের কণ্ঠে ডাবিং চাইলে **ঐচ্ছিক** ক্লোনিং (IndicF5)।
- 🔑 **কোনো লুকানো প্রম্পট/টার্মিনাল প্রশ্ন নেই** — key সেলের ভেতরেই পেস্ট করবেন; না দিলে সাথে সাথে পরিষ্কার error, আর HF token না দিলে ক্লোনিং নিজে থেকেই বাদ পড়বে।
- 🩺 **আর ঝুলে থাকবে না** — অডিও extract করার সময় লাইভ শতাংশ দেখাবে; কোনো অগ্রগতি না হলে নিজেই থেমে software decoding-এ নামবে বা স্পষ্ট error দেবে।
- ✂️ **নীরবতা-সচেতন chunking** — Whisper আর কখনো শব্দের মাঝখানে কাটে না (`XYZ` → `XY`+`Z` সমস্যা শেষ)।
- 🔁 **Rolling prompt + seam repair** — বাক্যের ধারাবাহিকতা ও punctuation ঠিক থাকে।
- ⏱️ **Duration-aware অনুবাদ (Gemini)** — প্রতিটি লাইনকে তার সময়ের বাজেট জানানো হয়।
- 🚫 **কোনো overlap নেই** — দুই কণ্ঠ কখনো একসাথে কথা বলে না (hard guarantee)।
- ⚡ **২–৩ মিনিটে ১০ মিনিটের ভিডিও** — অনুবাদ ও TTS একসাথে চলে।

### শুরুর আগে কেবল দুটি জিনিস দরকার

| দরকার | কোথায় |
|---|---|
| ভিডিও (mp4/mkv/mov/avi) | `MyDrive/Video/input/` |
| **Gemini API key** (অনুবাদের জন্য) | ধাপ ৬-এর সেলে পেস্ট করবেন (অথবা Colab Secrets-এ `GEMINI_API_KEY`) |

> **ঐচ্ছিক:** নিজের গলায় ডাবিং চাইলে `Video/input/`-এ ১০–২০ সেকেন্ডের বাংলা
> কথার audio (wav/mp3/m4a) রাখুন আর ধাপ ৫-এ আপনার HF token বসান। না করলে অ্যাপ
> নিজেই বিল্ট-ইন ভয়েস দিয়ে কাজ শেষ করবে — কিছুই ভাঙবে না, কিছুই জিজ্ঞেস করবে না।


## 🖥️ ধাপ ১: GPU যাচাই

আউটপুটে **Tesla T4** দেখা উচিত। Whisper `large-v3` এখানে দ্রুত চলে, আর নিজের
কণ্ঠ ক্লোন করতে চাইলে GPU কার্যত বাধ্যতামূলক।

`Runtime → Change runtime type → T4 GPU` সেট করা আছে কি না দেখে নিন।


In [ ]:
!nvidia-smi

## 📂 ধাপ ২: Google Drive mount

Drive-এ থাকা ভিডিও ও কণ্ঠের নমুনা পড়তে এবং ফলাফল সেভ করতে Drive যুক্ত করা হচ্ছে।

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

INPUT_DIR = '/content/drive/MyDrive/Video/input'
OUTPUT_DIR = '/content/drive/MyDrive/Video/output'

import os
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Input :', INPUT_DIR)
print('Output:', OUTPUT_DIR)

## ⬇️ ধাপ ৩: রিপোজিটরি clone করা

আগে থেকে clone করা থাকলে সেটি আপডেট (pull) করা হবে — নতুন কোড নিজে থেকেই চলে আসবে।

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/redwan-primeAmax/ProXMax.git'
REPO_DIR = '/content/ProXMax'


def _git(*args):
    return subprocess.run(
        ['git', '-C', REPO_DIR, *args], capture_output=True, text=True, check=False
    )


if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('আগের clone পাওয়া গেছে — সর্বশেষ কোডে আপডেট করা হচ্ছে…')
    _git('fetch', '--depth', '1', 'origin', 'main')
    if _git('merge', '--ff-only', 'origin/main').returncode != 0:
        print('স্থানীয় পরিবর্তন থাকায় fast-forward হলো না — জোর করে সর্বশেষ কোডে আনা হচ্ছে…')
        _git('reset', '--hard', 'origin/main')
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

print('বর্তমান commit:', _git('log', '-1', '--oneline').stdout.strip())


## 📦 ধাপ ৪: Dependency ইনস্টল

`requirements.txt` = মূল পাইপলাইন (Whisper, Gemini, বিল্ট-ইন বাংলা ভয়েস)।
কণ্ঠ **ক্লোনিং** চাইলে অতিরিক্ত `requirements-indicf5.txt` লাগবে — সেটি ধাপ ৫-এর
ঐচ্ছিক সেলে নিজে থেকেই ইনস্টল হবে।

> ইনস্টল ১–২ মিনিট নিতে পারে — শুধু প্রথমবার।


In [ ]:
%cd /content/ProXMax
!pip install -q -r requirements.txt


In [ ]:
import shutil
import subprocess

# ffmpeg না থাকলে ইনস্টল করা (Colab-এ সাধারণত আগে থেকেই থাকে)
if shutil.which('ffmpeg') is None:
    print('ffmpeg পাওয়া যায়নি — ইনস্টল করা হচ্ছে…')
    subprocess.run('apt-get -qq update && apt-get -qq install -y ffmpeg', shell=True, check=True)

print('ffmpeg:', shutil.which('ffmpeg'))
import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 🎤 ধাপ ৫: ঐচ্ছিক — নিজের কণ্ঠে ডাবিং (voice cloning)

**এটি না করলেও সব কাজ চলবে।** ডিফল্টে বিল্ট-ইন বাংলা neural ভয়েস
(`bn-BD-NabanitaNeural`) ব্যবহার হয় — কোনো টোকেন, অনুমতি বা রেকর্ডিং লাগে না।

নিজের গলায় ডাবিং চাইলে নিচের সেলে দুটি কাজ:

1. `Video/input/`-এ ১০–২০ সেকেন্ডের পরিষ্কার বাংলা কথার audio রাখুন।
2. `HF_TOKEN = ''` এর ভেতরে আপনার Hugging Face token পেস্ট করুন
   (https://huggingface.co/settings/tokens — `read` যথেষ্ট), আর
   https://huggingface.co/ai4bharat/IndicF5-এ **"Agree and access repository"**
   চেপে নিন (একবারই)। Colab Secrets-এ `HF_TOKEN` রাখলেও চলবে।

> **টোকেন ফাঁকা রাখলে** অ্যাপ ধরে নেবে আপনি ক্লোনিং চান না — কোনো ডাউনলোডের
> চেষ্টা করবে না, কোনো লগইন চাইবে না, সোজা বিল্ট-ইন ভয়েসে কাজ চালিয়ে যাবে।
> কোথাও কিছু টাইপ করতে হবে না। token কোথাও শেয়ার করবেন না।


In [ ]:
# ⬇️ ঐচ্ছিক: নিজের কণ্ঠ (cloning) — ফাঁকা রাখলে বিল্ট-ইন ভয়েসই চলবে
HF_TOKEN = ''          # ← চাইলে এখানে hf_... token পেস্ট করুন (নাহলে ফাঁকা রাখুন)

import glob
import os
import subprocess
import sys

CLONE_ENABLED = False
INPUT_DIR = globals().get('INPUT_DIR') or '/content/drive/MyDrive/Video/input'

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN') or ''
    except Exception:
        HF_TOKEN = os.environ.get('HF_TOKEN', '') or ''

sample_exts = ('.wav', '.mp3', '.m4a', '.flac')
samples = sorted(
    p for p in glob.glob(os.path.join(INPUT_DIR, '*'))
    if p.lower().endswith(sample_exts)
)

if not samples:
    print('ℹ️ Video/input-এ কণ্ঠের নমুনা নেই — বিল্ট-ইন বাংলা ভয়েস ব্যবহার হবে।')
    print('   কিছুই করতে হবে না, ধাপ ৬-এ যান।')
elif not HF_TOKEN.strip():
    print('ℹ️ HF token দেওয়া হয়নি — ধরে নেওয়া হলো আপনি ক্লোনিং চান না।')
    print('   বিল্ট-ইন বাংলা ভয়েস দিয়েই ডাবিং হবে; এখানে কিছু করতে হবে না।')
else:
    print('নমুনা পাওয়া গেছে:', [os.path.basename(p) for p in samples])
    print('IndicF5-এর জন্য অতিরিক্ত প্যাকেজ ইনস্টল করা হচ্ছে…')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-indicf5.txt'],
        cwd='/content/ProXMax', check=True,
    )
    from huggingface_hub import login, model_info
    token = HF_TOKEN.strip()
    login(token=token, add_to_git_credential=False)
    os.environ['HF_TOKEN'] = token
    try:
        print('✅ মডেল access ঠিক আছে:', model_info('ai4bharat/IndicF5').id)
        CLONE_ENABLED = True
    except Exception as exc:
        print('⚠️ IndicF5 মডেলে access নেই — ধাপ ৫-এর লিঙ্কে অনুমতি নিন।')
        print('   তবুও চলবে: বিল্ট-ইন ভয়েসে ডাবিং হবে।')
        print('   বিস্তারিত:', exc)

print('কণ্ঠ:', 'ক্লোনিং (IndicF5)' if CLONE_ENABLED else 'বিল্ট-ইন বাংলা ভয়েস')


## 🔑 ধাপ ৬: Gemini API key (একমাত্র বাধ্যতামূলক ধাপ)

অনুবাদ হয় **Google Gemini API** দিয়ে (ডিফল্ট মডেল `gemini-3.5-flash-lite`)।

- key নিন: https://aistudio.google.com/app/apikey
- নিচের সেলের `GEMINI_API_KEY = ''` এর ভেতরে **পেস্ট করে সেলটি রান করুন** —
  এরপর আর কোথাও কিছু দিতে হবে না, `Run all` চাপলে বাকি সব নিজে থেকে হবে।
- Colab Secrets-এ `GEMINI_API_KEY` রাখলে সেটিও নিজে থেকেই পড়া হবে।

> key না দিলে পরের ধাপে যাওয়ার আগেই **সাথে সাথে পরিষ্কার error** দেখাবে —
> পরে আটকে থেকে ঝুলে থাকবে না।


In [ ]:
# ⬇️ এখানে আপনার Gemini API key পেস্ট করুন (নাহলে Colab Secrets-এ GEMINI_API_KEY রাখুন)
GEMINI_API_KEY = ''

import os

if not GEMINI_API_KEY.strip():
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get('GEMINI_API_KEY') or ''
    except Exception:
        GEMINI_API_KEY = (
            os.environ.get('GEMINI_API_KEY', '') or os.environ.get('GOOGLE_API_KEY', '')
        )

GEMINI_API_KEY = GEMINI_API_KEY.strip()

if not GEMINI_API_KEY:
    raise SystemExit(
        '❌ Gemini API key পাওয়া যায়নি — কিছুই শুরু করা হলো না।\n'
        '   উপরের GEMINI_API_KEY = \'\' এর ভেতরে key পেস্ট করে এই সেলটি আবার রান করুন '
        '(অথবা Colab Secrets-এ GEMINI_API_KEY রাখুন)।\n'
        '   key নিন: https://aistudio.google.com/app/apikey'
    )

os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
print('✅ Gemini key সেট হয়েছে (শেষ ৪ অক্ষর: …%s)' % GEMINI_API_KEY[-4:])


## ⚙️ ধাপ ৭: দ্রুত সেটিংস (ঐচ্ছিক)

না বদলালেও চলবে — ডিফল্ট মানই সর্বোচ্চ গুণমানের। শুধু দরকার হলে বদলান।

| সেটিং | ডিফল্ট | কখন বদলাবেন |
|---|---|---|
| `WHISPER_MODEL` | `large-v3` | খুব ধীর লাগলে `medium` |
| `TTS_ENGINE` | `auto` | `auto` = নমুনা+টোকেন থাকলে ক্লোনিং, নাহলে বিল্ট-ইন। `edge` = সবসময় বিল্ট-ইন, `indicf5` = শুধু ক্লোনিং |
| `TTS_VOICE` | `bn-BD-NabanitaNeural` | পুরুষ কণ্ঠ চাইলে `bn-BD-PradeepNeural` |


In [ ]:
WHISPER_MODEL = 'large-v3'              # 'large-v3' (best) | 'medium' (faster)
TTS_ENGINE = 'auto'                     # 'auto' | 'indicf5' (ক্লোনিং) | 'edge' (বিল্ট-ইন)
TTS_VOICE = 'bn-BD-NabanitaNeural'      # পুরুষ: 'bn-BD-PradeepNeural'

args = []
if WHISPER_MODEL != 'large-v3':
    args += ['--model', WHISPER_MODEL]
if TTS_ENGINE != 'auto':
    args += ['--tts-engine', TTS_ENGINE]
if TTS_VOICE != 'bn-BD-NabanitaNeural':
    args += ['--voice', TTS_VOICE]
print('অতিরিক্ত আর্গুমেন্ট:', args or '(কিছু নেই — ডিফল্ট)')


## ▶️ ধাপ ৮: ডাবিং শুরু

`Video/input`-এর সব ভিডিও একটার পর একটা প্রসেস হবে। **নিচের সেলের আউটপুটে লগ
লাইভ (লাইন-বাইলাইন) আসবে** — কোনো লাইন একসাথে জমে থাকবে না।

প্রথম রানে শুধু Whisper মডেল (~৩ GB) ডাউনলোড হয় — তখন কয়েক মিনিট নতুন লগ
নাও দেখতে পারেন, কিন্তু প্রতি ৩০ সেকেন্ডে একটি heartbeat লাইন দেখবেন, তাই
"ঝুলে গেছে" ভেবে ভয় পাওয়ার কিছু নেই। ধাপ ৫-এ ক্লোনিং চালু করলে IndicF5
(~১.৪ GB)-ও নামবে।

লগে যা যা দেখবেন:

- `▶ ধাপ N/৭ …` — প্রতিটি ধাপের শুরু,
- `অডিও extract: 45% (04:30 / 10:00) — চলছে…` — extract-এর লাইভ অগ্রগতি,
- `STT 3/12 — 01:00–01:28` — কোন chunk ডিকোড হচ্ছে,
- `✓ ধাপ N/৭ শেষ …` এবং শেষে `✅ সম্পন্ন: …`।

কোনো ধাপ আটকে গেলে অনন্তকাল ঝুলে থাকবে না — নির্দিষ্ট সময় পর নিজেই থেমে
কারণ জানাবে।


In [ ]:
import os
import queue
import subprocess
import sys
import threading
import time

%cd /content/ProXMax

if not os.environ.get('GEMINI_API_KEY') and not os.environ.get('GOOGLE_API_KEY'):
    raise SystemExit('❌ আগে ধাপ ৬-এর সেলটি রান করুন (Gemini API key)।')

args = globals().get('args', [])          # ধাপ ৭ না চালালেও চলবে
command = [sys.executable, '-u', 'main.py'] + list(args)

print('চালানো হচ্ছে:', ' '.join(command), flush=True)
print('নিচে লগ লাইভ আসবে… (প্রথমবার Whisper মডেল নামাতে কয়েক মিনিট লাগতে পারে)\n', flush=True)

env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONIOENCODING='utf-8')
proc = subprocess.Popen(
    command,
    cwd='/content/ProXMax',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding='utf-8',
    errors='replace',
    bufsize=1,
    env=env,
)

lines: 'queue.Queue' = queue.Queue()


def _pump() -> None:
    assert proc.stdout is not None
    for line in proc.stdout:
        lines.put(line)
    lines.put(None)


threading.Thread(target=_pump, daemon=True).start()

printed = 0
last_output = time.monotonic()
while True:
    try:
        line = lines.get(timeout=1.0)
    except queue.Empty:
        # ৬০ সেকেন্ড চুপ থাকলে বোঝানো হয় কাজ এগোচ্ছে — আটকে যায়নি
        if time.monotonic() - last_output > 60:
            print('… এখনো কাজ চলছে (নতুন লগ নেই; Whisper মডেল ডাউনলোড/লোড হতে পারে)।',
                  flush=True)
            last_output = time.monotonic()
        continue
    if line is None:
        break
    print(line, end='', flush=True)
    printed += 1
    last_output = time.monotonic()

exit_code = proc.wait()
print('\n— শেষ: exit code %d —' % exit_code, flush=True)

if printed == 0:
    print('⚠️ কোনো লগ আসেনি। ধাপ ৪ (pip install) রান করেছেন কি না দেখুন, '
          'তারপর এই সেলটি আবার চালান।', flush=True)
elif exit_code != 0:
    print('ℹ️ উপরের লগে ❌ চিহ্নিত লাইনটিই আসল কারণ।', flush=True)


## 🎉 ধাপ ৯: ফলাফল দেখা

প্রতিটি ভিডিওর নামের শেষে `_bn.mp4` — সেটাই ডাবিং করা ভিডিও। পাশে `_en.srt` (মূল) ও `_bn.srt` (বাংলা) সাবটাইটেল।

In [ ]:
import os
import subprocess

files = sorted(os.listdir(OUTPUT_DIR)) if os.path.isdir(OUTPUT_DIR) else []
videos = [f for f in files if f.lower().endswith('.mp4')]

print(f'মোট ফাইল: {len(files)} | ডাবিং করা ভিডিও: {len(videos)}\n')
for name in files:
    path = os.path.join(OUTPUT_DIR, name)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f'  {name:45s} {size_mb:7.1f} MB')

if videos:
    print('\nDrive থেকে সরাসরি দেখা/ডাউনলোড করা যাবে:', os.path.join(OUTPUT_DIR, videos[0]))

In [ ]:
# প্রথম ডাবিং করা ভিডিওটি ব্রাউজারে চালিয়ে শোনা (Drive থেকে)
from IPython.display import HTML, display
import os

videos = [f for f in sorted(os.listdir(OUTPUT_DIR)) if f.lower().endswith('_bn.mp4')]
if videos:
    path = os.path.join(OUTPUT_DIR, videos[0])
    display(HTML(f'<video controls width="640" src="/content/drive/MyDrive/Video/output/{videos[0]}"></video>'))
    print('ভিডিও:', path)
else:
    print('কোনো ডাবিং করা ভিডিও পাওয়া যায়নি — ধাপ ৮-এর লগ দেখুন।')

## 🧹 ধাপ ১০: GPU মেমরি ছেড়ে দেওয়া

ডাবিং শেষে প্রোগ্রাম নিজেই মডেল আনলোড করে GPU ছেড়ে দেয়। তবু নিশ্চিত করতে এই সেলটি রান করুন (Colab সেশন চালু রাখলেও VRAM খালি থাকবে)।

In [ ]:
import gc

for name in ('pipeline', 'model', 'tts_engine'):
    globals()[name] = None
gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        used = torch.cuda.memory_allocated() / (1024 ** 3)
        print(f'GPU মেমরি এখন ব্যবহৃত: {used:.2f} GB (খালি করা হয়েছে)')
    else:
        print('GPU পাওয়া যায়নি — CPU মোড।')
except Exception as exc:
    print('torch ছাড়াই পরিষ্কার করা হলো:', exc)

## 🩺 সমস্যা হলে

| লক্ষণ | কারণ ও সমাধান |
|---|---|
| ধাপ ৮-এ কিছুই দেখা যাচ্ছে না | লগ এখন লাইভ আসে; নিচের সেলটি `python -u` দিয়ে চালায়। তবু কিছু না এলে ধাপ ৪ (`pip install -r requirements.txt`) রান করে আবার চেষ্টা করুন |
| কোড পুরোনো (নতুন ফিচার নেই) | ধাপ ৩-এর সেল আবার রান করুন — এখনই সর্বশেষ `main` commit এনে দেখায় |
| `Gemini API key পাওয়া যায়নি` | ধাপ ৬-এর `GEMINI_API_KEY = ''`-এর ভেতরে key পেস্ট করে সেলটি রান করুন |
| ক্লোনিং হচ্ছে না (বিল্ট-ইন ভয়েসে চলছে) | ধাপ ৫-এর `HF_TOKEN` খালি — চাইলে token বসান; HF অনুমতি নেওয়া আছে কি না দেখুন। এটা কোনো error নয় |
| `edge-tts` ব্যর্থ / কণ্ঠ আসছে না | ইন্টারনেট সংযোগ দেখুন; `pip install -U edge-tts`; `TTS_VOICE` বৈধ `bn-BD-*` কি না দেখুন |
| অডিও extract-এ আটকে যায় | এখন আর ঝুলে থাকে না: লাইভ শতাংশ দেখাবে, নির্দিষ্ট সময় অগ্রগতি না হলে থেমে software decoding-এ নামবে বা স্পষ্ট error দেবে। তবু সমস্যা হলে ভিডিওটি আবার আপলোড করুন (Drive-এ অর্ধেক আপলোড হওয়া ফাইল প্রায়ই পড়া যায় না) |
| `CUDA out of memory` | Runtime → Restart, ধাপ ৮ আবার চালান; নোটবুক নিজেই TTS-এর আগে Whisper আনলোড করে |
| `Gemini API error (429)` | quota শেষ — কিছুক্ষণ অপেক্ষা করুন; কোড নিজেই ৭ বার retry করে |
| `ffmpeg পাওয়া যায়নি` | ধাপ ৪-এর ffmpeg সেলটি রান করুন |
| ডাবিং ধীর লাগছে | `WHISPER_MODEL='medium'` করুন; এবং `config.yaml`-এর `tts.time_budget_factor` কমানো যাবে |

বিস্তারিত: রিপোর `README.md`।
